# Train a YOLO Nano Cube Detector in Colab

This notebook turns the corrected CVAT/COCO annotations into an Ultralytics YOLO detection dataset and trains a nano model for one class: `cube`.

Upload these two files when prompted:

- `adjustedAnnotations.zip` from CVAT, containing `annotations/instances_default.json`
- `cube_bbox_bootstrap_cvat_images.zip`, the image zip used to create the CVAT task

Use a GPU runtime: **Runtime -> Change runtime type -> T4 GPU** or better.


## 1. Install and Check GPU


In [ ]:
!nvidia-smi
!pip -q install -U ultralytics pyyaml


In [ ]:
from pathlib import Path
import json
import random
import shutil
import zipfile

from google.colab import files

WORKDIR = Path("/content/cube_yolo")
RAW_DIR = WORKDIR / "raw"
DATASET_DIR = WORKDIR / "dataset"
ANNOTATIONS_DIR = WORKDIR / "annotations"
IMAGES_DIR = WORKDIR / "images_source"

for directory in [RAW_DIR, DATASET_DIR, ANNOTATIONS_DIR, IMAGES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(WORKDIR)


## 2. Upload Corrected Annotations and Images

Upload both files in the same prompt. If you keep them in Google Drive, you can replace this cell with a `drive.mount()` copy step.


In [ ]:
uploaded = files.upload()
for name, content in uploaded.items():
    target = RAW_DIR / name
    target.write_bytes(content)
    print(f"uploaded {target} ({target.stat().st_size / 1024 / 1024:.1f} MB)")


## 3. Extract and Locate COCO JSON + Images


In [ ]:
def unzip_to(zip_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination)

zip_paths = sorted(RAW_DIR.glob("*.zip"))
if not zip_paths:
    raise FileNotFoundError("Upload adjustedAnnotations.zip and cube_bbox_bootstrap_cvat_images.zip first.")

for zip_path in zip_paths:
    names = zipfile.ZipFile(zip_path).namelist()
    if any(name.endswith("instances_default.json") for name in names):
        print(f"extract annotations: {zip_path.name}")
        unzip_to(zip_path, ANNOTATIONS_DIR)
    else:
        print(f"extract images: {zip_path.name}")
        unzip_to(zip_path, IMAGES_DIR)

coco_candidates = sorted(ANNOTATIONS_DIR.rglob("instances_default.json"))
if not coco_candidates:
    raise FileNotFoundError("Could not find annotations/instances_default.json in the uploaded annotation zip.")
COCO_JSON = coco_candidates[0]
print("COCO JSON:", COCO_JSON)

image_exts = {".jpg", ".jpeg", ".png", ".webp"}
source_images = sorted(path for path in IMAGES_DIR.rglob("*") if path.suffix.lower() in image_exts)
print("source images:", len(source_images))
if not source_images:
    raise FileNotFoundError("Could not find images in the uploaded image zip.")


## 4. Convert COCO Boxes to YOLO Train/Val Layout

Ultralytics expects one normalized YOLO label file per image: `class x_center y_center width height`. This cell creates `dataset/images/{train,val}` and `dataset/labels/{train,val}`.


In [ ]:
VAL_FRACTION = 0.2
SEED = 17
CLASS_NAME = "cube"

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
for split in ["train", "val"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

coco = json.loads(COCO_JSON.read_text())
categories = {category["id"]: category["name"] for category in coco.get("categories", [])}
print("categories:", categories)

image_index = {}
for path in source_images:
    image_index[path.name] = path
    try:
        rel = path.relative_to(IMAGES_DIR).as_posix()
        image_index[rel] = path
    except ValueError:
        pass

annotations_by_image = {}
for annotation in coco.get("annotations", []):
    annotations_by_image.setdefault(annotation["image_id"], []).append(annotation)

images = list(coco.get("images", []))
random.Random(SEED).shuffle(images)
val_count = max(1, round(len(images) * VAL_FRACTION)) if len(images) > 1 else 0
val_ids = {image["id"] for image in images[:val_count]}

missing = []
copied = {"train": 0, "val": 0}
box_count = {"train": 0, "val": 0}

for image in images:
    file_name = image["file_name"]
    source = image_index.get(file_name) or image_index.get(Path(file_name).name)
    if source is None:
        missing.append(file_name)
        continue

    split = "val" if image["id"] in val_ids else "train"
    target_image = DATASET_DIR / "images" / split / source.name
    target_label = DATASET_DIR / "labels" / split / f"{source.stem}.txt"
    shutil.copy2(source, target_image)

    width = float(image["width"])
    height = float(image["height"])
    lines = []
    for annotation in annotations_by_image.get(image["id"], []):
        x, y, w, h = [float(value) for value in annotation["bbox"]]
        x = max(0.0, min(width, x))
        y = max(0.0, min(height, y))
        w = max(0.0, min(width - x, w))
        h = max(0.0, min(height - y, h))
        if w <= 0 or h <= 0:
            continue
        x_center = (x + w / 2.0) / width
        y_center = (y + h / 2.0) / height
        lines.append(f"0 {x_center:.6f} {y_center:.6f} {w / width:.6f} {h / height:.6f}")

    target_label.write_text("\n".join(lines) + ("\n" if lines else ""))
    copied[split] += 1
    box_count[split] += len(lines)

if missing:
    raise FileNotFoundError(f"Missing {len(missing)} images, first few: {missing[:5]}")

data_yaml = DATASET_DIR / "data.yaml"
data_yaml.write_text(
    f"path: {DATASET_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    f"  0: {CLASS_NAME}\n"
)

print("copied images:", copied)
print("boxes:", box_count)
print(data_yaml.read_text())


## 5. Sanity Check Labels

This renders a few converted YOLO boxes before training. If these look wrong, stop here and fix the dataset conversion before training.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def draw_yolo_boxes(image_path: Path, label_path: Path):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            cls, xc, yc, bw, bh = line.split()
            xc, yc, bw, bh = map(float, [xc, yc, bw, bh])
            x1 = (xc - bw / 2) * width
            y1 = (yc - bh / 2) * height
            x2 = (xc + bw / 2) * width
            y2 = (yc + bh / 2) * height
            draw.rectangle([x1, y1, x2, y2], outline="red", width=max(2, width // 500))
    return image

sample_images = sorted((DATASET_DIR / "images" / "train").glob("*"))[:6]
plt.figure(figsize=(14, 8))
for index, image_path in enumerate(sample_images, start=1):
    label_path = DATASET_DIR / "labels" / "train" / f"{image_path.stem}.txt"
    plt.subplot(2, 3, index)
    plt.imshow(draw_yolo_boxes(image_path, label_path))
    plt.axis("off")
    plt.title(image_path.name, fontsize=8)
plt.tight_layout()


## 6. Train YOLO Nano

`yolo11n.pt` is the default nano checkpoint here. If you prefer YOLOv8 nano, change `MODEL_NAME` to `yolov8n.pt`.


In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolo11n.pt"
EPOCHS = 80
IMGSZ = 640
BATCH = -1  # -1 lets Ultralytics choose an automatic batch size

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(WORKDIR / "runs"),
    name="cube_yolo_nano",
    patience=20,
    plots=True,
)


## 7. Validate and Preview Predictions


In [ ]:
best_model_path = WORKDIR / "runs" / "cube_yolo_nano" / "weights" / "best.pt"
print("best model:", best_model_path)

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(DATASET_DIR / "data.yaml"), imgsz=IMGSZ, device=0)
print(metrics)


In [ ]:
prediction_dir = WORKDIR / "predictions"
best_model.predict(
    source=str(DATASET_DIR / "images" / "val"),
    imgsz=IMGSZ,
    conf=0.25,
    save=True,
    project=str(prediction_dir),
    name="val_preview",
)

preview_paths = sorted((prediction_dir / "val_preview").glob("*.jpg"))[:6]
plt.figure(figsize=(14, 8))
for index, image_path in enumerate(preview_paths, start=1):
    plt.subplot(2, 3, index)
    plt.imshow(Image.open(image_path))
    plt.axis("off")
plt.tight_layout()


## 8. Download Weights


In [ ]:
files.download(str(best_model_path))
